# 03 — Huấn luyện, đánh giá và Grad-CAM

**Cell 8 → Cell 10**:

1. Train lần lượt **ResNet18 / DenseNet121 / EfficientNet-B0** với cùng recipe
   (AdamW 1e-4, batch 32, weighted CE, AMP, early stopping patience 5).
2. Đánh giá trên tập test: confusion matrix, FP/FN, bảng so sánh model.
3. Grad-CAM + sanity check cho hình minh hoạ trong báo cáo.

> Trên Colab: Runtime → T4 GPU. Trọng số tốt nhất lưu ở `results/models/`,
> lịch sử train ở `results/metrics/`.

## Cell 8 — Huấn luyện 3 kiến trúc

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch

from src.utils.config import load_project_configs
from src.data import build_dataloaders
from src.models import (
    Trainer,
    build_criterion,
    build_model,
    build_optimizer,
    build_scheduler,
    compute_class_weight,
    count_parameters,
)

configs = load_project_configs()
data_cfg, model_cfg, xai_cfg = configs["data"], configs["model"], configs["xai"]
train_cfg = model_cfg["training"]
CLASS_NAMES = data_cfg["classes"]["names"]
NUM_CLASSES = int(data_cfg["classes"]["num_classes"])

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")

split_dir = PROJECT_ROOT / data_cfg["paths"]["splits_dir"]
combined_cfg = {**data_cfg, **model_cfg}
loaders = build_dataloaders(
    split_dir / data_cfg["splits"]["train_csv"],
    split_dir / data_cfg["splits"]["val_csv"],
    split_dir / data_cfg["splits"]["test_csv"],
    config=combined_cfg,
)
print({name: len(loader.dataset) for name, loader in loaders.items()})

In [ ]:
results_dir = PROJECT_ROOT / data_cfg["paths"]["results_dir"]
checkpoint_dir = PROJECT_ROOT / train_cfg["checkpoint_dir"]
history_dir = PROJECT_ROOT / "results/metrics"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
history_dir.mkdir(parents=True, exist_ok=True)

runs = []

for model_entry in model_cfg["models"]:
    name = model_entry["name"]
    print(f"\n{'=' * 70}\n{name} - {count_parameters(build_model(name, NUM_CLASSES, pretrained=False)):,} tham số\n{'=' * 70}")

    model = build_model(
        name,
        num_classes=NUM_CLASSES,
        pretrained=bool(model_entry.get("pretrained", True)),
        dropout=float(model_entry.get("dropout", 0.0)),
    )

    class_weights = compute_class_weight(
        loaders["train"].dataset.targets,
        num_classes=NUM_CLASSES,
        mode=train_cfg["loss"]["class_weighting"],
    )
    criterion = build_criterion(train_cfg, class_weights=class_weights, device=device)
    optimizer = build_optimizer(model, train_cfg)
    scheduler = build_scheduler(optimizer, train_cfg, epochs=int(train_cfg["epochs"]))

    checkpoint_path = checkpoint_dir / model_cfg["checkpoints"][name]
    trainer = Trainer(
        model,
        loaders["train"],
        loaders["val"],
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=int(train_cfg["epochs"]),
        patience=int(train_cfg["early_stopping"]["patience"]),
        monitor=train_cfg["early_stopping"]["monitor"],
        mode=train_cfg["early_stopping"]["mode"],
        min_delta=float(train_cfg["early_stopping"]["min_delta"]),
        checkpoint_path=checkpoint_path,
        history_path=history_dir / f"train_history_{name}.csv",
        amp=bool(train_cfg["amp"]),
        grad_clip=float(train_cfg.get("grad_clip", 0.0)),
        num_classes=NUM_CLASSES,
        class_names=CLASS_NAMES,
    )

    started = time.time()
    history = trainer.fit()
    elapsed_min = (time.time() - started) / 60

    runs.append(
        {
            "model": name,
            "params_m": round(count_parameters(model) / 1e6, 2),
            "best_epoch": trainer.best_epoch,
            "best_val_f1_macro": round(float(trainer.early_stopping.best), 4),
            "train_minutes": round(elapsed_min, 1),
            "checkpoint": str(checkpoint_path.relative_to(PROJECT_ROOT)),
        }
    )
    display(history.tail(3))

runs_df = pd.DataFrame(runs)
display(runs_df)
runs_df.to_csv(results_dir / "metrics" / "training_summary.csv", index=False)

## Cell 9 — Đánh giá trên tập test

- Nạp lại trọng số tốt nhất cho từng model.
- Tính Accuracy / F1-macro / QWK / Sensitivity / Specificity.
- Vẽ confusion matrix và biểu đồ FP/FN.
- Xuất `results/metrics/model_comparison.csv` + danh sách ảnh sai.

In [ ]:
from torch.utils.data import DataLoader

from src.data import DRDataset
from src.evaluation import (
    build_error_dataframe,
    compute_metrics,
    fp_fn_table,
    plot_confusion_matrix,
    plot_fp_fn_chart,
    summarize_confusions,
)
from src.models import load_checkpoint, predict_loader

test_ds = DRDataset(
    split_dir / data_cfg["splits"]["test_csv"],
    image_size=int(data_cfg["image"]["size"]),
    class_names=CLASS_NAMES,
    return_meta=True,
)
test_loader = DataLoader(test_ds, batch_size=int(train_cfg["batch_size"]), shuffle=False, num_workers=0)

figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
comparison = []

for model_entry in model_cfg["models"]:
    name = model_entry["name"]
    checkpoint_path = checkpoint_dir / model_cfg["checkpoints"][name]
    if not checkpoint_path.is_file():
        print(f"Bỏ qua {name}: chưa có {checkpoint_path}")
        continue

    model = build_model(name, NUM_CLASSES, pretrained=False, dropout=float(model_entry.get("dropout", 0.0)))
    load_checkpoint(checkpoint_path, model=model, device=device)
    model.to(device).eval()

    y_true, y_pred, meta = predict_loader(
        model, test_loader, device=device, return_meta=True
    )
    metrics = compute_metrics(y_true, y_pred, num_classes=NUM_CLASSES, class_names=CLASS_NAMES)

    print(
        f"{name:>16} | acc {metrics['accuracy']:.4f} | F1-macro {metrics['f1_macro']:.4f} | "
        f"QWK {metrics['qwk']:.4f} | sens {metrics['sensitivity_macro']:.4f} | "
        f"spec {metrics['specificity_macro']:.4f}"
    )
    comparison.append(
        {
            "model": name,
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
            "qwk": metrics["qwk"],
            "sensitivity_macro": metrics["sensitivity_macro"],
            "specificity_macro": metrics["specificity_macro"],
        }
    )

    plot_confusion_matrix(
        metrics,
        class_names=CLASS_NAMES,
        out_path=figures_dir / f"confusion_matrix_{name}.png",
        title=f"Confusion matrix - {name} (test)",
    )

    errors = build_error_dataframe(
        y_true,
        y_pred,
        image_paths=[item["image_path"] for item in meta],
        patient_ids=[item["patient_id"] for item in meta],
        class_names=CLASS_NAMES,
    )
    errors.to_csv(results_dir / "metrics" / f"errors_{name}.csv", index=False)
    display(summarize_confusions(errors, top_n=5))

    fp_fn = fp_fn_table(y_true, y_pred, num_classes=NUM_CLASSES, class_names=CLASS_NAMES)
    plot_fp_fn_chart(
        fp_fn,
        out_path=figures_dir / f"fp_fn_chart_{name}.png",
        title=f"False Positive / False Negative - {name}",
    )

comparison_df = pd.DataFrame(comparison).sort_values("f1_macro", ascending=False)
display(comparison_df)
comparison_df.to_csv(results_dir / "metrics" / "model_comparison.csv", index=False)

## Cell 10 — Grad-CAM & sanity check

Target layer được đọc từ `configs/xai_config.yaml`:
ResNet18 → `layer4[-1]`, DenseNet121 → `features.denseblock4`,
EfficientNet-B0 → `features[-1]`.

Hình `gradcam_sanity_check.png` dùng để trả lời câu hỏi: *mô hình có nhìn vào
vùng tổn thương hay chỉ nhìn viền ảnh?*

In [ ]:
from src.models import resolve_target_layer
from src.xai import sanity_check

BEST_MODEL = "resnet18"  # đổi thành model có F1-macro cao nhất
checkpoint_path = checkpoint_dir / model_cfg["checkpoints"][BEST_MODEL]

model = build_model(BEST_MODEL, NUM_CLASSES, pretrained=False)
load_checkpoint(checkpoint_path, model=model, device=device)
model.to(device).eval()

gradcam_cfg = xai_cfg["gradcam"]
target_layer = resolve_target_layer(model, gradcam_cfg["target_layers"][BEST_MODEL])

rows = sanity_check(
    model,
    test_loader,
    target_layer,
    out_path=figures_dir / "gradcam_sanity_check.png",
    num_samples=int(gradcam_cfg["sanity_check"]["num_samples"]),
    class_names=CLASS_NAMES,
    device=device,
    alpha=float(gradcam_cfg["alpha"]),
    method=gradcam_cfg["method"],
)

for row in rows:
    print(
        f"true={CLASS_NAMES[row['label']]:<14} pred={CLASS_NAMES[row['prediction']]:<14} "
        f"path={row['image_path']}"
    )
print(f"\nĐã lưu: {figures_dir / 'gradcam_sanity_check.png'}")